# ValuePrism leakage controls

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeffVallyath/geometry-of-truth/blob/v1.0.0/notebooks/valueprism_leakage.ipynb)

## Why this experiment exists

The larger study asks whether a model represents how a consideration bears on an action in a particular situation, and ValuePrism supplies situations, considerations, and Supports or Opposes labels that can also reward repeated moral phrases instead of that relation. A phrase such as respect for privacy often carries a stable prior even when the relevant situation changes.

This notebook tests the dataset design before the activation experiment. The strict split places recognized consideration identities and situations on one side of the training and test boundary. The checkerboard endpoint uses the same two considerations in two situations where their labels reverse, causing any fixed preference for one consideration to cancel. These protections are complementary.

The main result of this review is methodological. Restoring held-out consideration identities raises a text-only paired-accuracy baseline by 7.294793 percentage points across five fixed draws, while restoring situations raises it by 0.526937 points. Phrase identity is therefore a material shortcut for this baseline.

## Start here

From GitHub, click the Open in Colab badge above. In Colab, choose Runtime and Run all. Demo mode verifies and presents public aggregate measurements. Full reconstruction retrieves ValuePrism after the dataset license has been accepted, reads HF_TOKEN from Colab Secrets, rebuilds the public measurements on CPU, and keeps licensed row text inside the active runtime.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PUBLIC_REPOSITORY = 'https://github.com/JeffVallyath/geometry-of-truth.git'
PUBLIC_REF = 'v1.0.0'
RUN_MODE = 'DEMO'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/geometry-of-truth")
    if (REPO_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", "tag", PUBLIC_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", PUBLIC_REF], check=True)
    elif not (REPO_ROOT / "pyproject.toml").is_file():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError("The Colab repository directory exists but is not a usable checkout")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", PUBLIC_REF, PUBLIC_REPOSITORY, str(REPO_ROOT)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(path for path in candidates if (path / "pyproject.toml").is_file())

SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

print({"mode": RUN_MODE})

In [ ]:
from IPython.display import display

from geometry_of_truth.leakage.contracts import load_bundle, number_lineage
from geometry_of_truth.leakage.plots import stress_test
from geometry_of_truth.leakage.results import (
    audit,
    candidate_supply,
    normalization,
    protections,
    split_lineage,
    stress_draws,
    sensitivity,
    uncertainty,
)

bundle = load_bundle(REPO_ROOT)
results = bundle["results"]
print("Artifact integrity verified")

## Candidate structure

A ValuePrism row pairs one situation with one consideration and labels that consideration Supports or Opposes. Removing the third Either label leaves 183,023 eligible rows. Among them, 20,032 situations contain at least one row of each valence, and 3,437 exact consideration phrases appear with both valences somewhere in the dataset.

Combining those reversals produces 13,923 possible checkerboards across 6,073 distinct consideration pairs. A checkerboard contains two situations and two considerations. The first consideration supports in one situation and opposes in the other, while the second consideration follows the opposite pattern. The 13,923 figure counts possible boards before semantic review, and the 6,073 figure counts unique pairs even when one pair supports several boards.

For scores f, the checkerboard interaction is

I = [f(s1,c1) - f(s1,c2)] + [f(s2,c2) - f(s2,c1)]

Any score formed by adding a situation term to a fixed consideration term gives I equal to zero. A nonzero reciprocal effect therefore requires sensitivity to the situation and consideration together.

In [ ]:
display(audit(results))
display(protections(results))

## Algorithmic identity grouping

Exact string matching misses spelling and wording variants. The grouping pipeline starts with 17,678 distinct raw forms. Case, punctuation, spacing, leading articles, and light plural normalization merge 1,971 variants and leave 15,707 forms. Removing standard Value, Right, and Duty prefixes merges another 1,106 and leaves 14,601 forms.

The final automatic stage represents each phrase with character fragments of length 3 through 5 and groups forms whose similarity reaches 0.85, with at most 25 forms in one cluster. It merges 4,410 additional forms and leaves 10,191 algorithmic identity clusters. The table column named collapsed from prior reports forms merged at that stage rather than dataset rows removed.

L3 is an automatic near-duplicate grouping. Human review decides whether borderline phrases express the same consideration in meaning and stakeholder role.

In [ ]:
display(normalization(results))

## Dataset lineage

Two branches begin from the same 183,023 binary rows and answer different questions. The audit branch keeps 164,279 rows attached to consideration identities that reverse somewhere, then keeps 110,656 rows in situations containing both labels. Those rows cover 19,068 situations and measure whether enough reciprocal structure exists for review.

The frozen split branch removes 421 identical rows from the original binary pool and leaves 182,602. It holds out 25 percent of the L3 clusters and 30 percent of situations. Their joint assignment produces 116,000 strict training rows and 7,394 strict test rows, while the remaining 59,208 rows touch the held-out side for only one of the two required identities and stay outside both strict partitions.

This branching explains why 116,000 training rows plus later categories can exceed the 110,656-row audit pool. The figures come from separate filters applied to the shared source rather than successive steps in one shrinking table.

In [ ]:
display(split_lineage(results))

## Deliberate shortcut restoration

The stress test fits the same logistic text classifier five times using seeds 0 through 4, with word unigrams and bigrams from the situation and consideration together as its features. Within-situation paired accuracy asks whether a Supports row receives a higher score than an Opposes row from the same situation, with ties worth one half. A score of 0.5 is chance and 1.0 is perfect ordering.

Each draw first uses the strict split. One intervention restores 30 percent of held-out consideration identities to training while the other restores situations, with test membership, classifier, features, and metric fixed within each draw. In the table, strict score is the baseline accuracy and each overlap score is the accuracy after its named restoration. The difference columns multiply that subtraction by 100, so they report percentage points rather than percent change.

Restoring consideration identities raises paired accuracy by 7.294793 points on average. Across the five draws, the sample standard deviation is 1.929464 points, the standard error is 0.862883, and the Student t 95 percent interval runs from 4.899047 to 9.690539. Restoring situations raises the mean by 0.526937 points. Its sample standard deviation is 1.072724, its standard error is 0.479737, and its interval runs from negative 0.805026 to 1.858899. The consideration interval stays above zero, while the situation interval includes zero.

The plotted lines connect strict and restored scores for the same seed. This result belongs to the text-only pair_text baseline and within-situation paired accuracy. Activation probes and checkerboard interactions use separate endpoints.

In [ ]:
display(stress_draws(results))
display(uncertainty(results))
display(stress_test(results))

## Stricter sensitivity sets

U0 is the 7,394-row strict test set. It contains 4,372 situations, 2,009 within-situation comparisons, and 378,506 cross-situation row pairs that share a consideration cluster. U1 removes the clearest remaining near-duplicate risks and retains 7,081 rows, 4,237 situations, 1,865 within-situation comparisons, and 374,160 within-consideration pairs in total.

U2 also removes every automatically ambiguous high-risk cluster. The set collapses to 587 rows in 547 situations, with 23 within-situation comparisons and 477 within-consideration pairs. The large within-consideration counts in U0 and U1 arise because every eligible row under one consideration can pair with rows from many other situations.

The U2 collapse shows where automatic exclusion loses the comparison structure needed for the relation test. Human adjudication is the next filter for borderline identity pairs.

In [ ]:
sensitivity_table = sensitivity(results)
display(sensitivity_table[sensitivity_table["set"].isin(["U0", "U1", "U2"])].reset_index(drop=True))

## Two human audits

The leakage audit asks whether exposure to a training phrase substantially defeats the intended holdout of a test phrase because both express the same consideration across splits. Reviewers see phrase pairs without labels, activation results, probe scores, or baseline performance.

The checkerboard audit examines a different unit. Reviewers decide whether each consideration keeps the same meaning and stakeholder role across both situations, then check whether all four labels support the reciprocal pattern. Separate records preserve the distinction between train-test identity control and checkerboard construct validity.

## Checkerboard supply

The ranked pool contains exactly 1,090 candidate checkerboards, and the planned endpoint needs 800 human-confirmed boards. Dividing 800 by 1,090 gives a required acceptance fraction of 0.733945, or 73.3945 percent.

The 1,090 figure is a census of the ranked pool, so sampling uncertainty does not apply to that count. Human acceptance remains unknown. Two independent reviewers first estimate it on a sample, and the resulting interval determines whether the pool can plausibly reach 800 accepted boards before the full audit proceeds.

In [ ]:
display(candidate_supply(results))

## Where every headline number comes from

Every displayed result comes from results.json after its integrity check passes. The final two columns below identify the exact field or calculation for each headline quantity. The fixed seeds, per-draw scores, split counts, sensitivity coverage, and candidate supply all remain available in that same public aggregate.

In [ ]:
display(number_lineage(bundle))

## Full reconstruction

Full mode installs the ValuePrism dependencies, reads HF_TOKEN from Colab Secrets, reconstructs the public aggregate on CPU, and compares all 13 headline quantities with the stored reference. This rerun is optional. Generated row-level files remain inside the active Colab runtime under the reconstruction directory.

In [ ]:
if RUN_MODE == "FULL":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[valueprism-full]"],
        check=True,
    )
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Add HF_TOKEN to Colab secrets and enable notebook access")
    from geometry_of_truth.leakage.reproduce import reproduce; full_run = reproduce("/content/valueprism-reproduction"); display(full_run["comparison"])
else:
    print("Set RUN_MODE to FULL and rerun from the first cell for raw reconstruction")

## Place in the larger experiment

The strict split measures generalization to unseen consideration identities and unseen situations. The reciprocal checkerboard measures whether a score changes with the relation after fixed consideration preferences cancel. Passing both controls would support a context-sensitive endorsement signal rather than phrase recognition alone.

Human review now determines the semantic quality of the retained identity boundaries and checkerboards. Its acceptance estimates set the usable sample size for the activation experiment and therefore determine whether the final relation endpoint has enough audited comparisons to justify interpretation. That audit comes next.